# Encoder swaps in a fixed BioLangFusion pipeline

We replace one pretrained DNA, RNA, or protein encoder at a time. The splits and training recipe come from the [Stage 3 BioLangFusion reference benchmarks](../stage3/blf_reference_benchmarks.ipynb).

The comparisons ask whether the replacement improves prediction, whether fusion adds value beyond the validation-selected single encoder, and whether a fusion head improves on concatenation with the same inputs. For the four-benchmark readout, see the [Newton three-seed findings](#Newton-three-seed-findings).

For the two additional RNA encoders tested on E. coli, see the [RNA encoder follow-up](#Replacing-RiNALMo-with-RNAErnie-and-ERNIE-RNA).


---

## Setup

You can read the saved tables as shown. To refresh them, keep `READ_SAVED_RESULTS = True` and the training and analysis switches off. Use the kernel described in the [notebook environment setup](../../README.md#notebook-dependencies).

To prepare or submit new fits, follow the [Newton execution guide](../../execution/arcc/encoder_swaps/README.md).


### Imports


In [1]:
# Files, environment, and elapsed time
import json
import os
import sys
import time
from pathlib import Path

# Set Apple GPU fallback before importing PyTorch; preserve an explicit user setting.
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

# Numerical arrays and result tables
import numpy as np
import pandas as pd

# Tensor operations, model training, and available compute backends
import torch

# Render notebook tables explicitly rather than relying on an implicit kernel name.
from IPython.display import display

### Project paths and shared code


In [2]:
# Search the working directory and its parents for Code/shared_code.
CODE_DIR = next(
    (
        candidate.resolve()
        for parent in (Path.cwd(), *Path.cwd().parents)
        for candidate in (parent, parent / "Code")
        if (candidate / "shared_code").is_dir()
    ),
    None,
)
if CODE_DIR is None:
    raise RuntimeError(
        "Run this notebook from the repository root or a directory inside it "
        "so Code/shared_code can be found."
    )
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

# Benchmark rows and encoder definitions
from shared_code import datasets
from shared_code.encoders import ENCODERS

# Fusion input streams, prediction-head training, and token-cache verification
from shared_code.fusion import blf, train, swap_results, swap_plan, run_identity
from shared_code.fusion import token_embeddings as tokens

### Compute device

Check the selected backend before starting extraction or training.


In [3]:
# Table display width (characters) and visible columns, not sequence or token limits.
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

# Preserve the existing preference: Apple MPS, then NVIDIA CUDA, then CPU.
DEVICE = (
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print(
    "torch", torch.__version__,
    "| CUDA available:", torch.cuda.is_available(),
    "| Apple MPS available:", torch.backends.mps.is_available(),
)
if DEVICE == "cuda":
    print(f"Selected device: {DEVICE} | GPU: {torch.cuda.get_device_name()}")
elif DEVICE == "mps":
    print(f"Selected device: {DEVICE} | GPU: Apple GPU (MPS)")
else:
    print(f"Selected device: {DEVICE}")

torch 2.10.0+cu126 | CUDA available: True | Apple MPS available: False
Selected device: cuda | GPU: NVIDIA GeForce RTX 2060 SUPER


### Settings and comparison scope

The notebook opens in saved-results mode so you can review the Newton comparisons without fitting again.

The four-benchmark study replaces one encoder at a time. The E. coli follow-up uses the same design with RNAErnie and ERNIE-RNA, repeating the RiNALMo comparison for gating and concatenation. Both sets of results appear in the default Newton readout.

The last cell selects the saved results and embedding locations. Set a workspace path there if your Newton results are stored outside the default results folder.


In [4]:
# Run mode
READ_SAVED_RESULTS = True
RUN_TRAINING = False
RUN_SAVED_COMPARISONS = False      # Build the report across COMPARISON_SEEDS from saved predictions.
RUN_SINGLE_SEED_INTERVALS = False  # Older calculation for SEED when READ_SAVED_RESULTS is False.

if RUN_SAVED_COMPARISONS and not READ_SAVED_RESULTS:
    raise ValueError(
        "Set READ_SAVED_RESULTS = True for saved-prediction reporting; "
        "training and extraction must remain disabled."
    )

In [5]:
# Comparison settings
EXPERIMENT = "encoder_swaps"

# The E. coli results table uses these replacements in either view.
ECOLI_RNA_SWAPS = (
    ("rna", "rnaernie"),
    ("rna", "ernierna"),
)

if EXPERIMENT in {"encoder_swaps", "full"}:  # "full" is the older Newton label.
    BENCHMARKS = (
        "cov_vac_degradation", "fungal_expression", "ecoli_proteins", "mrna_stability",
    )
    SWAPS = [
        ("dna", "nt500m"),
        ("dna", "hyenadna"),
        ("rna", "rinalmo"),
        ("rna", "mrnafm"),
        ("rna", "calm"),
        ("prot", "esm2_35m"),
        ("prot", "esm2_150m"),
        ("prot", "protbert"),
    ]
elif EXPERIMENT == "ecoli_ernie":
    # Repeat the RiNALMo comparison for gating and concatenation.
    BENCHMARKS = ("ecoli_proteins",)
    SWAPS = list(ECOLI_RNA_SWAPS)
    if RUN_SINGLE_SEED_INTERVALS or RUN_SAVED_COMPARISONS:
        raise ValueError("The focused panel reads head scores only; single-control reports are not configured.")
else:
    raise ValueError("EXPERIMENT must be 'encoder_swaps' or 'ecoli_ernie'.")

ERNIE_ENCODERS = tuple(key for _, key in ECOLI_RNA_SWAPS)
ORIGINAL = dict(blf.ENCODER_OF_MODALITY)

# Seeds and reporting
SEED = 0                       # Training seed for individual tables and local fits.
SELECTION_SEED = 0             # Use this fit seed's validation scores to select baselines and the MIL penalty.
COMPARISON_SEEDS = (0, 1, 2)    # Training seeds included in the saved-prediction report.
COMPARISON_DRAWS = 2000         # Number of test-sequence group resamples.
COMPARISON_BOOTSTRAP_SEED = 0  # Random seed for those resamples.

# Model and training settings
# The notebook also uses these settings to find matching saved fits.
TCONV = "equation"  # Equation 1 six-mer-to-codon upsampling.
CHANNELS = 100      # Learned filters per TextCNN kernel size.
BATCH_SIZE = 32
MAX_EPOCHS = 500    # Validation-based early stopping can end a fit sooner.

In [6]:
# Locations

# Saved results
RESULT_SET = "newton_h100_seeds012"

# Leave the workspace paths at None for results in this checkout.
# For moved results, point to the folder containing outputs/.
FOUR_BENCHMARK_WORKSPACE = None  # Defaults to newton_h100_seeds012.
ECOLI_FOLLOWUP_WORKSPACE = None  # Defaults to newton_ecoli_ernie.
SAVED_REPORT_FILES = {}     # One report path per benchmark when several reports match.

STAGE1_ALIGNMENT = CODE_DIR.parent / "docs" / "figures" / "stability" / "results.json"  # For the alignment comparison.

# Embedding caches
# Change these two paths if you store the embeddings on another drive.
CACHE_ROOT = CODE_DIR / "stage3" / "embeddings"
NEW_CACHE_ROOT = CODE_DIR / "stage4/embeddings/ecoli_ernie"  # RNAErnie and ERNIE-RNA.

# Cache format
CHUNK_SIZE = tokens.DEFAULT_CHUNK
DTYPE = np.float16  # Precision of stored cache arrays.

# Resolved paths and write protection
from shared_code.fusion.result_sets import resolve_swap_results, resolve_ernie_extension

if EXPERIMENT == "ecoli_ernie":
    RESULT_PATHS = resolve_ernie_extension(
        CODE_DIR, workspace=ECOLI_FOLLOWUP_WORKSPACE,
        reference_workspace=FOUR_BENCHMARK_WORKSPACE,
    )
    REFERENCE_SWAP_ROOT = resolve_swap_results(
        CODE_DIR, "newton_h100_seeds012", workspace=FOUR_BENCHMARK_WORKSPACE,
    ).swaps
else:
    RESULT_PATHS = resolve_swap_results(
        CODE_DIR, RESULT_SET, workspace=FOUR_BENCHMARK_WORKSPACE,
    )
    REFERENCE_SWAP_ROOT = RESULT_PATHS.swaps
if not READ_SAVED_RESULTS:
    RESULT_PATHS.require_writable_fits()
print(f"Result set: {RESULT_PATHS.name}; evidence root: {RESULT_PATHS.root}")

RESULTS_ROOT = RESULT_PATHS.swaps
ORIGINAL_RESULTS = RESULT_PATHS.originals
COMPARISON_DIR = RESULT_PATHS.analysis_dir

# The Newton launcher supplies fit provenance and checkpoint checks.
CACHE_REUSE_ONLY = EXPERIMENT == "ecoli_ernie"
CHECKPOINT_VALIDATOR = None
FIT_PRODUCER = None

Result set: newton_h100_seeds012; evidence root: D:\academic\UCF\2026_fall\Capstone Project CAP6942\Research\genomics-multimodal-fusion\Code\stage4\results\newton_h100_seeds012


#### Fit variants and embedding caches

The four-benchmark study uses the same swap list for each benchmark. The selected experiment determines which fit variants the notebook looks for. Fits on the same benchmark can reuse an encoder’s embeddings across heads and training seeds. The E. coli follow-up reuses the existing DNA and protein embeddings and stores the two additional RNA caches separately.


In [7]:
def requested_variants(modality, encoder_key):
    """List the fit variants for one replacement encoder."""
    if EXPERIMENT == "ecoli_ernie":
        return ("mil", "concat")
    return swap_results.swap_variants(
        modality, grid=tokens.grid_resolution(encoder_key) != "codon",
    )


# The metadata inventory covers every encoder used by the selected experiment.
inventory_originals = (
    (ORIGINAL["dna"], ORIGINAL["prot"])
    if EXPERIMENT == "ecoli_ernie" else ORIGINAL.values()
)
INVENTORY_ENCODERS = list(dict.fromkeys([
    *inventory_originals,
    *(encoder_key for _, encoder_key in SWAPS),
]))


def cache_directory(benchmark, encoder_key):
    """Find the cache for this benchmark and encoder."""
    root = CACHE_ROOT
    if EXPERIMENT == "ecoli_ernie" and encoder_key in ERNIE_ENCODERS:
        root = NEW_CACHE_ROOT
    return root / benchmark / encoder_key


def cache_is_read_only(encoder_key):
    """The E. coli follow-up may extract only its two additional RNA embeddings."""
    return EXPERIMENT == "ecoli_ernie" and encoder_key not in ERNIE_ENCODERS


## Benchmarks and targets

This study uses the four available BioLangFusion benchmarks. The fifth, Ab1, was absent from the release checked in the [Stage 3 benchmark audit](../stage3/blf_datasets.ipynb#The-five-benchmarks-BioLangFusion-reports).

Each retained row pairs a nucleotide sequence with a supplied target and split. We keep the supplied train, validation, and test assignments and drop rows with sequences longer than 999 nucleotides. The cutoff follows the [Stage 3 reconstruction of the length filter](../stage3/blf_datasets.ipynb#Recovering-BioLangFusion's-preprocessing).

| Benchmark | What the model predicts |
| --- | --- |
| [CoV-Vac](../../docs/methods/data-and-targets.md#cov-vac-chemical-degradation-of-designed-rna) | Chemical degradation of designed RNA under the lab assay |
| [Fungal expression](../../docs/methods/data-and-targets.md#fungal-expression) | Log RNA-expression score from RNA-seq |
| [E. coli expression](../../docs/methods/data-and-targets.md#protein-expression-in-e-coli) | Low, medium, or high protein-expression category from gel scores |
| [mRNA stability](../../docs/methods/data-and-targets.md#mrna-stability-standardized-decay-profiles) | Relative stability within the source experimental profile |


### From one supplied sequence to three encoder streams

The nucleotide sequence from each retained row supplies all three encoder inputs. DNA and RNA encoders read the same sequence in their respective alphabets, using `U` in place of `T` for RNA. The protein input is computed by translating complete triplets from the first supplied base. Stops become `X` so those triplets keep their positions. For CoV-Vac's designed RNA fragments, the reading frame is a modeling assumption. The alphabet and translation rules are implemented in the [shared input preparation code](../shared_code/fusion/token_embeddings.py).

The encoders retain a feature vector at each token position. Before fusion, the **codon grid** maps the different token resolutions to one position per complete triplet. The table below shows how each replacement reaches that grid.


### What changes and what stays fixed

The reference uses BioLangFusion's original NT v2 100M multi-species for DNA, RNA-FM for RNA, and ESM-2 8M for protein. Each row replaces one encoder while keeping the other two, benchmark rows and splits, codon grid, fusion and prediction settings, and scoring fixed. Each replacement brings its own weights, feature width, and tokenization, so the results describe the effect of the whole swap.

In [8]:
planned = len(BENCHMARKS) * len(SWAPS)
print(f"{planned} swap-benchmark pairs over {len(BENCHMARKS)} benchmarks, device {DEVICE}")
pd.DataFrame([
    {
        "slot": modality,
        "replaces": ENCODERS[ORIGINAL[modality]].label,
        "swapped in": ENCODERS[encoder_key].label,
        "grid": tokens.grid_resolution(encoder_key),
    }
    for modality, encoder_key in SWAPS
])

32 swap-benchmark pairs over 4 benchmarks, device cuda


,slot,replaces,swapped in,grid
0,dna,NT v2 100M multi-species,NT 500M human-ref,kmer
1,dna,NT v2 100M multi-species,HyenaDNA large,nucleotide
2,rna,RNA-FM,RiNALMo 150M,nucleotide
3,rna,RNA-FM,mRNA-FM,codon
4,rna,RNA-FM,CaLM,codon
5,prot,ESM-2 8M,ESM-2 35M,codon
6,prot,ESM-2 8M,ESM-2 150M,codon
7,prot,ESM-2 8M,ProtBERT,codon


**Why these replacements.** CaLM, mRNA-FM, ProtBERT, and alternative DNA encoders were candidates in the [original Phase B plan](../../docs/Multimodal_Research%20Plan%20.docx). RiNALMo and ESM-2 35M extend the panel to replacements whose pooled representations were similar to those of the original encoders in [Stage 1](../stage1/Stage1_stability.ipynb#Final-layer-representation-similarity:-linear-CKA). Those swaps let us ask whether substituting a similar representation preserves prediction performance and how it changes the gain from fusion.

DNABERT-2's variable-length tokens require a separate grid adapter, an open item in the [Stage 4 runbook](RUNBOOK.md#remaining-dependencies-and-deferred-work).

#### Plan the selected seed

In the four-benchmark study, each swap expands into single-encoder controls and fusion heads. The E. coli follow-up requests only MIL and concatenation. The plan checks the requested fits at `SEED`, including any original-encoder fits required for comparison.

The first table counts fits by benchmark, role, and status. The second lists fits that need work or review. New and resumable fits determine which embedding caches need checking.

Saved-results mode leaves embedding arrays and checkpoints unopened. In computation mode, every benchmark must pass planning before extraction starts. If a checkpoint needs inspection, the notebook checks its recorded settings and recovery state on CPU.


In [9]:
def result_recipe(seed):
    """Return the training settings used to match a saved fit, including defaults."""
    return vars(train.TrainConfig(
        batch_size=BATCH_SIZE,
        max_epochs=MAX_EPOCHS,
        channels=CHANNELS,
        tconv=TCONV,
        seed=seed,
    ))

# Plan every requested fit before extraction. Reuse actual saved paths, including
# compatible historical names; a missing result is different from a conflicting file.
def requested_identity(benchmark, variant, config, mapping, *, rows=None, verification=None):
    """Describe the ordered rows, encoders, and cache metadata this fit must match."""
    if rows is None:
        rows = datasets.blf_benchmark_rows(benchmark)
    spec = datasets.DATASETS[benchmark]
    records = {
        modality: run_identity.cache_record(
            cache_directory(benchmark, encoder_key), verification=verification,
        )
        for modality, encoder_key in mapping.items()
    }
    return run_identity.blf_identity(
        rows=rows, benchmark=benchmark, variant=variant, config=config,
        encoders=mapping, caches=records, task=spec.task,
        label_column=spec.label_column, split_column=spec.split_column,
    )

def execution_plan(benchmark):
    """Use the same preflight before extraction and immediately before fitting."""
    focused = EXPERIMENT == "ecoli_ernie"
    return swap_plan.plan_benchmark(
        benchmark=benchmark,
        config=result_recipe(SEED),
        selection_seed=SELECTION_SEED,
        original_encoders=ORIGINAL,
        original_folders=ORIGINAL_RESULTS[benchmark],
        original_destination=ORIGINAL_RESULTS[benchmark][-1],
        # In the standard panel, a true third field adds a grid single-encoder control.
        swaps=[
            (modality, encoder_key, tokens.grid_resolution(encoder_key) != "codon")
            for modality, encoder_key in SWAPS
        ],
        swap_root=RESULTS_ROOT / benchmark,
        selection_original_folders=(
            RESULT_PATHS.selection_originals[benchmark]
            if RESULT_PATHS.fixed_historical_selection else None
        ),
        selection_snapshot=RESULT_PATHS.selection_snapshot,
        selection_swap_root=(
            RESULT_PATHS.selection_swaps / benchmark
            if RESULT_PATHS.fixed_historical_selection else None
        ),
        checkpoint_validator=(
            None if READ_SAVED_RESULTS else CHECKPOINT_VALIDATOR or train.checkpoint_preflight
        ),
        identity_factory=None if READ_SAVED_RESULTS else requested_identity,
        fit_variants=("mil", "concat") if focused else None,
        include_originals=not focused,
        select_singles=not focused,
    )

In [10]:
FIT_PLANS = {benchmark: execution_plan(benchmark) for benchmark in BENCHMARKS}
fit_inventory = pd.DataFrame([row for plan in FIT_PLANS.values() for row in plan.rows()])
if not fit_inventory.empty:
    display(fit_inventory.groupby(["benchmark", "role", "status"]).size().rename("fits"))
    display(fit_inventory.loc[fit_inventory["status"] != "reuse"])
for benchmark, plan in FIT_PLANS.items():
    for issue in plan.issues:
        print(f"{benchmark}: {issue}")

if not READ_SAVED_RESULTS:
    # Stop the entire request before expensive work if any comparison is blocked.
    for plan in FIT_PLANS.values():
        plan.require_ready()

PENDING_RUNS = [
    run for plan in FIT_PLANS.values() for run in plan.runs
    if run.status in ("new", "resume")
]
# Check a cache once, in the order it is first needed by a pending fit.
PENDING_CACHE_KEYS = list(dict.fromkeys(
    (run.benchmark, encoder_key)
    for run in PENDING_RUNS for encoder_key in run.encoders.values()
))
print(f"Training seed {SEED}; selection seed {SELECTION_SEED}; "
      f"{len(PENDING_RUNS)} new/resumable fits in the displayed plan.")

benchmark            role         status
cov_vac_degradation  original     reuse      7
                     replacement  reuse     35
ecoli_proteins       original     reuse      7
                     replacement  reuse     35
fungal_expression    original     reuse      7
                     replacement  reuse     35
mrna_stability       original     reuse      7
                     replacement  reuse     35
Name: fits, dtype: int64

,benchmark,role,encoder,variant,seed,lambda,status,reason,source,destination,legacy identity


Training seed 0; selection seed 0; 0 new/resumable fits in the displayed plan.


### Plan disk space for replacement caches

The fit plan identifies which caches the unfinished fits need. This estimates the full size of each replacement encoder's token embeddings for the benchmark's retained sequences, including embeddings already on disk. The [cache checks](#Check-caches-before-loading-models) identify which chunks still need extraction.


In [11]:
# Token-array payload only; widths describe the selected checkpoints.
WIDTHS = {
    "nt500m": 1280,
    "hyenadna": 256,
    "rinalmo": 640,
    "rnaernie": 768,
    "ernierna": 768,
    "mrnafm": 1280,
    "calm": 768,
    "esm2_35m": 480,
    "esm2_150m": 640,
    "protbert": 1024,
}


def estimate_cache_storage(benchmarks, pending_cache_keys, encoder_widths, dtype):
    """Estimate full token-array sizes for replacement caches used by pending fits."""
    pending_benchmarks = {
        benchmark for benchmark, _ in pending_cache_keys
    }
    bytes_per_feature = np.dtype(dtype).itemsize
    storage_rows = []

    for benchmark in benchmarks:
        if benchmark not in pending_benchmarks:
            continue

        sequence_lengths = datasets.blf_benchmark_rows(benchmark)["length"].to_numpy()

        for _, encoder_key in SWAPS:
            if (benchmark, encoder_key) not in pending_cache_keys:
                continue

            encoder = ENCODERS[encoder_key]
            token_counts = np.array([
                tokens.tokens_for_length(encoder, int(length))
                for length in sequence_lengths
            ])
            # Content tokens x features/token x bytes/feature; decimal GB.
            # This excludes model and training memory.
            gigabytes = (
                token_counts.sum()
                * encoder_widths[encoder_key]
                * bytes_per_feature
                / 1e9
            )
            storage_rows.append({
                "benchmark": benchmark,
                "encoder": encoder.label,
                "rows": len(sequence_lengths),
                "GB": round(gigabytes, 1),
            })

    return pd.DataFrame(
        storage_rows,
        columns=["benchmark", "encoder", "rows", "GB"],
    )


In [12]:
projection = pd.DataFrame()

if READ_SAVED_RESULTS:
    print("Storage projection skipped: reading saved results does not load benchmark inputs.")
elif not PENDING_RUNS:
    print("All requested fits are complete; no extraction storage is needed.")
else:
    projection = estimate_cache_storage(
        BENCHMARKS,
        PENDING_CACHE_KEYS,
        WIDTHS,
        DTYPE,
    )
    display(
        projection.pivot_table(
            index="benchmark",
            columns="encoder",
            values="GB",
            aggfunc="sum",
        ).fillna("")
    )
    print(
        "Estimated replacement-cache payload needed by pending fits, including existing caches: "
        f"~{projection['GB'].sum():.0f} GB"
    )
    print("Cache arrays stay out of Git; cache_directory selects the configured roots.")


Storage projection skipped: reading saved results does not load benchmark inputs.


### Check which saved fits can be compared

The four-benchmark study checks original and replacement fits against the same training settings for seeds 0, 1, and 2. The E. coli follow-up checks its requested fits at the selected seed. For each benchmark, the gated head reuses the entropy penalty chosen on seed-0 validation with the original encoders.

A fit can have a saved score and still lack the paired predictions needed for an interval. **Legacy identity** counts fits whose encoder mapping is inferred from the results folder. Files from other seeds are expected when seeds share a folder. The exclusion summary shows other reasons, with full details in `result_issues`.

In [13]:
def result_variant_label(result):
    """Keep unregularized MIL separate from the selected positive-penalty variant."""
    return (
        "mil (lam 0)"
        if result.variant == "mil" and result.config["entropy_lambda"] == 0
        else result.variant
    )


def original_report(benchmark, seed):
    return swap_results.read_results(
        ORIGINAL_RESULTS[benchmark],
        benchmark=benchmark,
        config=result_recipe(seed),
        encoders=ORIGINAL,
        native_fallback=True,
    )


def original_runs(benchmark, seed=None):
    """Read the selected seed; ambiguous identities cannot silently replace one another."""
    report = original_report(benchmark, SEED if seed is None else seed)
    # Duplicated identities are withheld by the reader; other saved scores still stand.
    records = report.records if READ_SAVED_RESULTS else report.require_unique()
    return [train.Result.load(record["path"]) for record in records]


def selected_mil_penalty(benchmark):
    """Reuse the complete original positive-penalty grid's seed-0 selection."""
    mil_penalty = FIT_PLANS[benchmark].lambda_
    if mil_penalty is None:
        raise RuntimeError(f"{benchmark}: seed-0 MIL selection unavailable; inspect the execution plan")
    return mil_penalty


In [14]:
def swap_report(benchmark, modality, encoder_key, seed, mil_penalty):
    """Check the replacement recipe, encoder mapping when recorded, and fit names."""
    expected = requested_variants(modality, encoder_key)
    return swap_results.read_results(
        [RESULTS_ROOT / benchmark / encoder_key],
        benchmark=benchmark,
        config=result_recipe(seed),
        encoders={**ORIGINAL, modality: encoder_key},
        mil_lambda=mil_penalty,
        variants=(
            expected if mil_penalty is not None
            else tuple(variant for variant in expected if variant != "mil")
        ),
    )

In [16]:
def coverage_for_benchmark(benchmark, seeds):
    """Collect saved-fit coverage and exclusions for one benchmark and its references."""
    coverage_rows, issue_rows = [], []
    original_variants = swap_plan.ORIGINAL_VARIANTS

    try:
        inherited_lambda = selected_mil_penalty(benchmark)
        prerequisite = ""
    except RuntimeError as exc:
        inherited_lambda = None
        prerequisite = str(exc)

    # Zero here marks unavailable positive-penalty selection; it does not select MIL(0).
    coverage_lambda = inherited_lambda if inherited_lambda is not None else 0.0
    for inspected_seed in seeds:
        if EXPERIMENT == "ecoli_ernie":
            for modality, encoder_key in SWAPS:
                report = swap_report(benchmark, modality, encoder_key, inspected_seed, inherited_lambda)
                coverage_rows.append({
                    "benchmark": benchmark, "encoder": ENCODERS[encoder_key].label, "seed": inspected_seed,
                    **report.coverage(("mil", "concat"), mil_lambda=coverage_lambda),
                    "reference gaps": "not fitted in this panel", "reference arrays absent": "",
                    "prerequisite": prerequisite,
                })
                issue_rows.extend({"benchmark": benchmark, "encoder": encoder_key,
                                   "seed": inspected_seed, **issue} for issue in report.issues)
            continue
        originals = original_report(benchmark, inspected_seed)
        # The coverage row concerns the inherited penalty, not another seed's tuning winner.
        original_coverage = originals.coverage(original_variants, mil_lambda=coverage_lambda)
        coverage_rows.append({
            "benchmark": benchmark, "encoder": "original", "seed": inspected_seed,
            **original_coverage, "reference gaps": "", "reference arrays absent": "",
            "prerequisite": prerequisite,
        })
        issue_rows.extend({
            "benchmark": benchmark, "encoder": "original", "seed": inspected_seed, **issue,
        } for issue in originals.issues)

        for modality, encoder_key in SWAPS:
            expected = requested_variants(modality, encoder_key)
            report = swap_report(
                benchmark, modality, encoder_key, inspected_seed, inherited_lambda,
            )
            references = [
                *(
                    f"unimodal:{reference_modality}"
                    for reference_modality in ORIGINAL if reference_modality != modality
                ),
                *(("unimodal-grid:rna",) if modality != "rna" else ()),
                "concat", "mil", "cross",
            ]
            reference_coverage = originals.coverage(references, mil_lambda=coverage_lambda)
            coverage_rows.append({
                "benchmark": benchmark, "encoder": ENCODERS[encoder_key].label,
                "seed": inspected_seed,
                **report.coverage(expected, mil_lambda=coverage_lambda),
                "reference gaps": reference_coverage["missing"],
                "reference arrays absent": reference_coverage["without paired arrays"],
                "prerequisite": prerequisite,
            })
            issue_rows.extend({
                "benchmark": benchmark, "encoder": encoder_key, "seed": inspected_seed, **issue,
            } for issue in report.issues)
    return coverage_rows, issue_rows


In [17]:
# Inspection only: these seeds do not alter SEED or launch additional fits.
COVERAGE_SEEDS = (SEED,) if EXPERIMENT == "ecoli_ernie" else (0, 1, 2)
coverage_rows, issue_rows = [], []
for benchmark in BENCHMARKS:
    benchmark_coverage, benchmark_issues = coverage_for_benchmark(benchmark, COVERAGE_SEEDS)
    coverage_rows.extend(benchmark_coverage)
    issue_rows.extend(benchmark_issues)

result_coverage = pd.DataFrame(coverage_rows).set_index(["benchmark", "encoder", "seed"])
result_issues = pd.DataFrame(issue_rows)
print("Gated-head entropy penalties from seed-0 validation")
for benchmark in BENCHMARKS:
    penalty = FIT_PLANS[benchmark].lambda_
    penalty_text = f"{penalty:g}" if penalty is not None else "unavailable"
    print(f"  {benchmark}  {penalty_text}")

display(result_coverage.drop(columns="excluded files"))
if not result_issues.empty:
    exclusions_to_review = result_issues.loc[
        result_issues["reason"] != "configuration differs: seed"
    ]
    if not exclusions_to_review.empty:
        display(
            exclusions_to_review.groupby(["seed", "reason"]).size()
            .rename("excluded records")
        )

available missing without paired arrays ambiguous  excluded files  legacy identity reference gaps  \
benchmark           encoder           seed                                                                                                                                              
cov_vac_degradation original          0     unimodal:dna, unimodal:rna, unimodal:prot, uni...                                                      14                0                  
                    NT 500M human-ref 0     unimodal:dna, unimodal-grid:dna, concat, mil, ...                                                      10                0                  
                    HyenaDNA large    0     unimodal:dna, unimodal-grid:dna, concat, mil, ...                                                      10                0                  
                    RiNALMo 150M      0     unimodal:rna, unimodal-grid:rna, concat, mil, ...                                                      10                0                  
                    mRNA-FM           0                      unimodal:rna, concat, mil, cross                                                       8                0                  
...                                                                                       ...     ...                   ...       ...             ...              ...            ...   
mrna_stability      mRNA-FM           2                      unimodal:rna, concat, mil, cross                                                       8                0                  
                    CaLM              2                      unimodal:rna, concat, mil, cross                                                       8                0                  
                    ESM-2 35M         2                     unimodal:prot, concat, mil, cross                                                       8                0                  
                    ESM-2 150M        2                     unimodal:prot, concat, mil, cross                                                       8                0                  
                    ProtBERT          2                     unimodal:prot, concat, mil, cross                                                       8                0                  

                                           reference arrays absent prerequisite  
benchmark           encoder           seed                                       
cov_vac_degradation original          0                                          
                    NT 500M human-ref 0                                          
                    HyenaDNA large    0                                          
                    RiNALMo 150M      0                                          
                    mRNA-FM           0                                          
...                                                            ...          ...  
mrna_stability      mRNA-FM           2                                          
                    CaLM              2                                          
                    ESM-2 35M         2                                          
                    ESM-2 150M        2                                          
                    ProtBERT          2                                          

[108 rows x 9 columns]

seed  reason                     
0     configuration differs: seed    336
1     configuration differs: seed    336
2     configuration differs: seed    336
Name: excluded records, dtype: int64

---

## Part 1: Prepare the token embeddings

### Check caches before loading models

The fit plan above identifies which encoder caches the pending fits need. In saved-results mode, the table reads cache metadata only, so the recorded shard counts can be shown even if the embedding files are unavailable. In computation mode, the check matches the cache settings and sequence order and inspects the stored shards for the pending fits.

Missing shards can pass this check when extraction is allowed. Incompatible caches and incomplete reuse-only caches stop execution before any encoder loads. The table identifies the available and missing shards and explains any blockers.


In [18]:
def saved_cache_status(benchmark, encoder_key):
    """Read one metadata status row without opening or verifying token shards."""
    path = cache_directory(benchmark, encoder_key) / "meta.json"
    try:
        metadata = json.loads(path.read_text(encoding="utf-8"))
        if not isinstance(metadata, dict):
            raise ValueError("metadata is not an object")
        cache_status = "saved metadata; shards not checked"
        issue = ""
    except (OSError, ValueError) as exc:
        metadata = {}
        cache_status = "metadata unavailable"
        issue = str(exc)
    return {
        "benchmark": benchmark,
        "encoder": ENCODERS[encoder_key].label,
        "status": cache_status,
        "recorded shards": metadata.get("n_shards"),
        "issues": issue,
    }


if READ_SAVED_RESULTS:
    # Inspect small metadata files only. Their presence does not verify the shards.
    CACHE_PLANS = {}
    BENCHMARK_ROWS = {}
    checks = [
        saved_cache_status(benchmark, encoder_key)
        for benchmark in BENCHMARKS
        for encoder_key in INVENTORY_ENCODERS
    ]
    display(pd.DataFrame(checks).set_index(["benchmark", "encoder"]))
    print("Saved scores remain readable without token caches. No recovery is attempted.")


status  recorded shards issues
benchmark           encoder                                                                             
cov_vac_degradation NT v2 100M multi-species  saved metadata; shards not checked               10       
                    RNA-FM                    saved metadata; shards not checked               10       
                    ESM-2 8M                  saved metadata; shards not checked               10       
                    NT 500M human-ref         saved metadata; shards not checked               10       
                    HyenaDNA large            saved metadata; shards not checked               10       
                    RiNALMo 150M              saved metadata; shards not checked               10       
                    mRNA-FM                   saved metadata; shards not checked               10       
                    CaLM                      saved metadata; shards not checked               10       
                    ESM-2 35M                 saved metadata; shards not checked               10       
                    ESM-2 150M                saved metadata; shards not checked               10       
                    ProtBERT                  saved metadata; shards not checked               10       
fungal_expression   NT v2 100M multi-species  saved metadata; shards not checked               13       
                    RNA-FM                    saved metadata; shards not checked               13       
                    ESM-2 8M                  saved metadata; shards not checked               13       
                    NT 500M human-ref         saved metadata; shards not checked               13       
                    HyenaDNA large            saved metadata; shards not checked               13       
                    RiNALMo 150M              saved metadata; shards not checked               13       
                    mRNA-FM                   saved metadata; shards not checked               13       
                    CaLM                      saved metadata; shards not checked               13       
                    ESM-2 35M                 saved metadata; shards not checked               13       
                    ESM-2 150M                saved metadata; shards not checked               13       
                    ProtBERT                  saved metadata; shards not checked               13       
ecoli_proteins      NT v2 100M multi-species  saved metadata; shards not checked               18       
                    RNA-FM                    saved metadata; shards not checked               18       
                    ESM-2 8M                  saved metadata; shards not checked               18       
                    NT 500M human-ref         saved metadata; shards not checked               18       
                    HyenaDNA large            saved metadata; shards not checked               18       
                    RiNALMo 150M              saved metadata; shards not checked               18       
                    mRNA-FM                   saved metadata; shards not checked               18       
                    CaLM                      saved metadata; shards not checked               18       
                    ESM-2 35M                 saved metadata; shards not checked               18       
                    ESM-2 150M                saved metadata; shards not checked               18       
                    ProtBERT                  saved metadata; shards not checked               18       
mrna_stability      NT v2 100M multi-species  saved metadata; shards not checked               94       
                    RNA-FM                    saved metadata; shards not checked               94       
                    ESM-2 8M                  saved metadata; shards not checked               94       
                    NT 500M human-ref         saved metadata; shards not checked    

Saved scores remain readable without token caches. No recovery is attempted.


In [19]:
if not READ_SAVED_RESULTS:
    # Historical shard evidence is optional; ordered-input identity is still required.
    recovery_path = CACHE_ROOT / "recovery-records.json"
    RECOVERY_RECORDS = (
        {
            recovery_record["key"]: recovery_record
            for recovery_record in json.loads(recovery_path.read_text())["caches"]
        }
        if recovery_path.exists() else {}
    )

    # Share successful file checks with extraction until the file statistics change.
    CACHE_VERIFICATION = tokens.CacheVerification()
    CACHE_PLANS = {}
    BENCHMARK_ROWS = {}
    checks = []
    for benchmark, encoder_key in PENDING_CACHE_KEYS:
        if benchmark not in BENCHMARK_ROWS:
            BENCHMARK_ROWS[benchmark] = datasets.blf_benchmark_rows(benchmark)
        rows = BENCHMARK_ROWS[benchmark]
        encoder = ENCODERS[encoder_key]
        texts = tokens.inputs_for_rows(encoder, rows["sequence"].tolist())
        plan = tokens.inspect_cache(
            encoder,
            texts,
            cache_directory(benchmark, encoder_key),
            chunk_size=CHUNK_SIZE,
            dtype=DTYPE,
            recovery_evidence=RECOVERY_RECORDS.get(f"{benchmark}/{encoder_key}"),
            verification=CACHE_VERIFICATION,
        )
        CACHE_PLANS[benchmark, encoder_key] = plan
        checks.append({
            "benchmark": benchmark,
            "encoder": encoder.label,
            "status": plan.status,
            "reusable shards": len(plan.reusable_shards),
            "missing shards": len(plan.missing_shards),
            "input provenance": plan.provenance,
            "issues": "; ".join(plan.issues),
        })
        print(f"  {benchmark}/{encoder_key}: {plan.status}, {plan.provenance}", flush=True)
    if checks:
        display(pd.DataFrame(checks).set_index(["benchmark", "encoder"]))
    else:
        print("All requested fits are complete; cache verification is unnecessary.")

    # Stop before loading models if any cache cannot safely be reused or extended.
    blocked = [
        plan.summary for (_, encoder_key), plan in CACHE_PLANS.items()
        if not plan.ready or (
            (CACHE_REUSE_ONLY or cache_is_read_only(encoder_key))
            and (plan.missing_shards or plan.restore_metadata)
        )
    ]
    if blocked:
        raise tokens.CacheRecoveryRequired(
            "Cache preflight stopped before model loading:\n" + "\n".join(blocked)
        )
    missing = sum(len(plan.missing_shards) for plan in CACHE_PLANS.values())
    print(f"Preflight passed: {len(CACHE_PLANS)} caches; {missing} shards need extraction.")


### Extract missing shards and reuse complete caches

A shard stores token embeddings for a chunk of sequences. Extraction fills the allowed gaps from the cache check. Completed shards can be reused after an interruption. The E. coli follow-up requires complete DNA and protein caches and uses separate caches for the two RNA replacements.


In [20]:
written = []
if READ_SAVED_RESULTS:
    print("Extraction skipped. Missing caches or shards are not computed in read-saved mode.")
elif CACHE_REUSE_ONLY:
    print("Complete caches verified; extraction and metadata repair disabled for this execution.")
else:
    started = time.perf_counter()
    for benchmark, encoder_key in PENDING_CACHE_KEYS:
        if cache_is_read_only(encoder_key):
            continue  # Verified above; never extract or repair the existing DNA/protein caches.
        encoder = ENCODERS[encoder_key]
        rows = BENCHMARK_ROWS[benchmark]
        texts = tokens.inputs_for_rows(encoder, rows["sequence"].tolist())
        cache, wrote_shards = tokens.extract(
            encoder, texts, cache_directory(benchmark, encoder_key), DEVICE,
            chunk_size=CHUNK_SIZE, dtype=DTYPE,
            recovery_evidence=RECOVERY_RECORDS.get(f"{benchmark}/{encoder_key}"),
            verification=CACHE_VERIFICATION,
        )
        try:
            written.append({
                "benchmark": benchmark, "encoder": encoder.label, "sequences": len(cache),
                "tokens": cache.meta["total_tokens"], "width": cache.width,
                "GB": round(cache.meta["bytes"] / 1e9, 2),
                "new shards written": "yes" if wrote_shards else "no",
            })
        finally:
            cache.close()
    if written:
        display(pd.DataFrame(written).set_index(["benchmark", "encoder"]))
    print(f"Extraction/reuse checks finished in {(time.perf_counter() - started) / 60:.1f} minutes.")


Extraction skipped. Missing caches or shards are not computed in read-saved mode.


The verified caches supply the encoder streams for the pending fits below.


---

## Part 2: Fit the encoder comparisons

### Prediction heads and controls

The encoders stay frozen. We fit TextCNN to predict the target from a single stream or a learned combination of all three. Its filters read three, four, or five positions, so a filter spans different lengths at native resolution and on the codon grid. The [TextCNN implementation](../shared_code/fusion/textcnn.py) shows the filters and pooling used in both cases.

| Fit | What the predictor uses | What the comparison tells us |
| --- | --- | --- |
| Native single (`unimodal:<slot>`) | One encoder at its own token resolution | How well that encoder predicts on its own |
| Codon-grid single (`unimodal-grid:<slot>`) | One encoder placed on the codon grid | How changing the resolution affects the single-encoder score |
| Concatenation (`concat`) | All three streams joined and projected at each grid position | The same-input reference for gating and cross-attention |
| MIL gating (`mil`) | All three streams mixed using one weight per modality for the entire sequence | Whether adapting modality weights helps |
| Cross-attention (`cross`) | Each stream attends to the joint context of all three streams before merging | Whether exchanging information across positions and streams helps |

The entropy penalty for MIL favors concentrating weight on fewer modalities. The [fusion heads](../shared_code/fusion/heads.py) show where those weights are applied.


### Open the embedding streams for each fit group

A fit group shares one encoder combination. The code opens and checks its three caches once for the group, then closes them when the fits finish or an error interrupts the group. Groups whose fits are all saved leave the caches unopened.


In [21]:
def open_fit_streams(benchmark, encoder_mapping):
    """Open one triple, releasing earlier caches if a later open/check fails."""
    rows = BENCHMARK_ROWS.get(benchmark)
    if rows is None:
        rows = datasets.blf_benchmark_rows(benchmark)
    spec = datasets.DATASETS[benchmark]
    caches = {}
    try:
        for modality, encoder_key in encoder_mapping.items():
            caches[modality] = tokens.summarize(cache_directory(benchmark, encoder_key))
        streams = blf.BenchmarkStreams(
            benchmark, rows, caches, spec.task, spec.label_column, spec.split_column,
            encoders=dict(encoder_mapping),
        )
        streams.check()
        return streams
    except BaseException as error:
        for cache in caches.values():
            try:
                cache.close()
            except BaseException as cleanup_error:
                if hasattr(error, "add_note"):
                    error.add_note(f"Cache cleanup also failed: {cleanup_error}")
        raise


In [22]:
def run_pending_group(benchmark, encoder_key, planned_runs):
    """Run only new/resumable fits for one triple; reuse needs no token arrays."""
    pending = [run for run in planned_runs if run.status in ("new", "resume")]
    if not pending:
        return
    streams = open_fit_streams(benchmark, pending[0].encoders)
    primary_error = None
    try:
        for run in pending:
            print(f"{benchmark} / {encoder_key or 'original references'} / {run.variant} / seed {SEED}")
            train.run(
                streams, run.variant, train.TrainConfig(**run.config), DEVICE,
                run.path.parent, tag=run.path.stem,
                run_identity=requested_identity(
                    benchmark, run.variant, run.config, run.encoders, rows=streams.rows,
                    verification=CACHE_VERIFICATION,
                ),
                producer=(
                    FIT_PRODUCER(benchmark) if FIT_PRODUCER is not None
                    else run_identity.producer_record(device=DEVICE, repository=CODE_DIR.parent)
                ),
            )
    except BaseException as error:
        primary_error = error
        raise
    finally:
        try:
            streams.close()
        except BaseException as cleanup_error:
            if primary_error is None:
                raise
            if hasattr(primary_error, "add_note"):
                primary_error.add_note(f"Stream cleanup also failed: {cleanup_error}")

### Fit the selected seed

Recheck the plan after extraction so fits completed in the meantime can be reused. Pending original-encoder fits run first. In the four-benchmark study, the native and grid singles, concatenation, and MIL run before cross-attention. In the E. coli follow-up, both gated fits run before either concatenation fit.

An interrupted fit can resume from its saved checkpoint when the recovery checks pass. See the [encoder-swap continuation guide](RUNBOOK.md#complete-the-encoder-swaps-one-seed-at-a-time) for checkpoint and result-file handling.


In [23]:
if READ_SAVED_RESULTS:
    print("Training skipped. The execution plan reports reusable, missing and blocked fits.")
elif not RUN_TRAINING:
    print("Training skipped: extraction-only execution; no fit was started.")
else:
    started = time.perf_counter()
    # Recheck destinations after extraction, so a completed canonical or renamed
    # result discovered since preflight is also reused rather than fitted again.
    FIT_PLANS = {benchmark: execution_plan(benchmark) for benchmark in BENCHMARKS}
    for plan in FIT_PLANS.values():
        plan.require_ready()

    # New seeds need same-seed original references as well as replacement fits.
    # Their missing entries are explicit in the plan, never inferred as completed.
    for benchmark, plan in FIT_PLANS.items():
        original_runs_to_fit = [run for run in plan.runs if run.role == "original"]
        run_pending_group(benchmark, None, original_runs_to_fit)

    if EXPERIMENT == "ecoli_ernie":
        # Finish both gated fits before either concatenation fit.
        for variant in ("mil", "concat"):
            for benchmark, plan in FIT_PLANS.items():
                for _, encoder_key in SWAPS:
                    requested = [run for run in plan.runs
                                 if run.encoder_key == encoder_key and run.variant == variant]
                    run_pending_group(benchmark, encoder_key, requested)
    else:
        # Preserve the original sweep's cheap-first, cross-attention-second order.
        for run_cross_attention in (False, True):
            for benchmark, plan in FIT_PLANS.items():
                for modality, encoder_key in SWAPS:
                    replacement_runs = [
                        run for run in plan.runs
                        if run.role == "replacement"
                        and run.encoder_key == encoder_key
                        and (run.variant == "cross") == run_cross_attention
                    ]
                    run_pending_group(benchmark, encoder_key, replacement_runs)
    print(f"Requested seed finished in {(time.perf_counter() - started) / 3600:.2f} hours.")


Training skipped. The execution plan reports reusable, missing and blocked fits.


---

## Part 3: Interpret the comparisons

Each comparison asks what changed when we replaced an encoder, added fusion, or changed the fusion head.


### Compare each swap with its baselines

`alone` is the swapped encoder's native-resolution score. The selected single can be any eligible stream in that encoder combination, including a codon-grid control. Seed-0 validation fixes that choice. Each row uses the test score of the selected reference from the same training seed.

- **Fusion gain** compares a head with that combination's selected single.
- **Gain over original** compares the same head before and after the encoder swap.
- **Head comparison** uses concatenation with the same inputs as the reference for MIL or cross-attention.

The regression benchmarks use Spearman correlation. E. coli scores are accuracy percentages and score differences are in percentage points.


In [24]:
# E. coli accuracy is displayed as a percentage; score differences are percentage points.
SCALE = {"ecoli_proteins": 100.0}
HEADS = swap_results.HEADS


def swap_runs(benchmark, encoder_key, mil_penalty, seed=None):
    """Read only the selected seed and inherited recipe, refusing duplicate identities."""
    modality = next(
        slot for slot, replacement_encoder in SWAPS
        if replacement_encoder == encoder_key
    )
    report = swap_report(
        benchmark, modality, encoder_key, SEED if seed is None else seed, mil_penalty,
    )
    records = report.records if READ_SAVED_RESULTS else report.require_unique()
    return [train.Result.load(result["path"]) for result in records]


def results_for_encoder_set(benchmark, modality=None, encoder_key=None, seed=None):
    """Collect unique variant records for the original or a replacement triple."""
    try:
        mil_penalty = selected_mil_penalty(benchmark)
    except RuntimeError:
        if not READ_SAVED_RESULTS:
            raise
        # Missing MIL selection does not hide independent single/concat/cross scores.
        mil_penalty = None
    runs = {}
    # Keep the validation-selected positive MIL penalty among the original runs.
    for result in original_runs(benchmark, seed):
        name = result_variant_label(result)
        if name == "mil" and result.config["entropy_lambda"] != mil_penalty:
            continue
        runs[name] = result
    if encoder_key is None:
        return runs

    # Retain unchanged-slot singles; replace this slot's baselines and compared heads.
    for name in [
        variant for variant in runs
        if variant in HEADS or variant.endswith(f":{modality}")
    ]:
        del runs[name]
    for result in swap_runs(benchmark, encoder_key, mil_penalty, seed):
        runs[result_variant_label(result)] = result
    return runs


In [25]:
def validation_selected_single(runs, modality=None, encoder_key=None, selection_runs=None, *, benchmark=None):
    """Select on validation only when the required single-encoder references exist."""
    if EXPERIMENT == "ecoli_ernie":
        return None  # No new single-control panel has been fitted or selected.
    if benchmark is not None and RESULT_PATHS.fixed_historical_selection:
        selected = FIT_PLANS[benchmark].selected_singles.get(encoder_key or "original")
        return selected if selected in runs else None
    required = {f"unimodal:{slot}" for slot in ORIGINAL}
    if encoder_key and tokens.grid_resolution(encoder_key) != "codon":
        required.add(f"unimodal-grid:{modality}")
    selection_runs = runs if selection_runs is None else selection_runs
    if not required.issubset(selection_runs):
        return None
    # Available historical grid controls remain eligible; missing native references
    # must not make an incomplete set look like a fully compared "best single."
    singles = {
        variant: result for variant, result in selection_runs.items()
        if variant.startswith("unimodal")
    }
    selected_variant = max(singles, key=lambda variant: singles[variant].val_score)
    return selected_variant if selected_variant in runs else None

In [26]:
def extension_head_scores(benchmark, directory, encoder_key, seed, penalty):
    """Read the two head accuracies for one encoder and training seed."""
    report = swap_results.read_results(
        [directory / benchmark / encoder_key], benchmark=benchmark,
        config=result_recipe(seed), encoders={**ORIGINAL, "rna": encoder_key},
        mil_lambda=penalty, variants=("mil", "concat"),
    )
    return {entry["data"]["variant"]: 100 * entry["data"]["test_score"]
            for entry in report.require_unique()}


def extension_score_table(extension_root, seeds):
    """Read matched E. coli heads from the extension and the Newton reference."""
    benchmark = "ecoli_proteins"
    penalty = selected_mil_penalty(benchmark)
    rows = []
    for encoder_key, directory, evidence in [
        ("rinalmo", REFERENCE_SWAP_ROOT, "existing RiNALMo reference"),
        *((key, extension_root, "focused extension") for key in ERNIE_ENCODERS),
    ]:
        scores = {seed: extension_head_scores(benchmark, directory, encoder_key, seed, penalty)
                  for seed in seeds}
        for variant, head in (("mil", "MIL gated"), ("concat", "Concatenation")):
            row = {"RNA encoder": ENCODERS[encoder_key].label, "head": head, "evidence": evidence}
            for seed in seeds:
                row[f"Seed {seed} (%)"] = scores[seed].get(variant, np.nan)
            rows.append(row)
    table = pd.DataFrame(rows).set_index(["RNA encoder", "head", "evidence"])
    if tuple(seeds) == (0, 1, 2):
        table["Mean (%)"] = table.mean(axis=1, skipna=False)
    return table


def benchmark_score_table(benchmark):
    """Build one benchmark's score and gain rows from saved results."""
    scale = SCALE.get(benchmark, 1.0)
    original = results_for_encoder_set(benchmark)
    rows = []
    # The (None, None) entry supplies the original-triple reference row first.
    for modality, encoder_key in [(None, None), *SWAPS]:
        runs = results_for_encoder_set(benchmark, modality, encoder_key)
        selection_runs = (
            runs if SEED == SELECTION_SEED
            else results_for_encoder_set(
                benchmark, modality, encoder_key, seed=SELECTION_SEED,
            )
        )
        selected_single = validation_selected_single(
            runs, modality, encoder_key, selection_runs, benchmark=benchmark,
        )
        # No replacement-only score exists for the original row; unavailable scores stay NaN.
        row = {
            "slot": modality or "none",
            "encoder": (
                ENCODERS[encoder_key].label if encoder_key else "BioLangFusion's own"
            ),
            "alone": (
                runs.get(f"unimodal:{modality}").test_score * scale
                if modality and f"unimodal:{modality}" in runs else np.nan
            ),
            "best single": selected_single,
            "best single score": (
                runs[selected_single].test_score * scale if selected_single else np.nan
            ),
        }
        for head in HEADS:
            score = runs[head].test_score * scale if head in runs else np.nan
            row[head] = score
            # Distinguish fusion benefit from the overall effect of replacing an encoder.
            row[f"{head} Δfusion"] = score - row["best single score"]
            row[f"{head} vs original"] = (
                score - original[head].test_score * scale
                if head in original and encoder_key else np.nan
            )
        rows.append(row)
    return pd.DataFrame(rows).set_index(["slot", "encoder"])

### Compare models across training seeds

In saved-results mode, this step reads matching reports. To calculate a report from saved predictions and benchmark labels, use the [report instructions](RUNBOOK.md#run-focused-reports-and-cpu-probes). The report also compares the replacement single with the original single. Each contrast needs paired predictions for every seed in `COMPARISON_SEEDS` to report a complete mean. The mean averages the fitted models' scores. For individual contrasts by seed, see the report JSON identified below.

Seed spread is the standard deviation of the paired differences across training seeds. The 95% paired interval uses 2,000 resamples of test-sequence groups in the Newton reports, conditional on those fitted seeds and without an adjustment for multiple comparisons. If the interval spans zero, the direction of the difference remains unresolved under this procedure.


In [28]:
def build_or_reuse_report(benchmark, reporting):
    """Calculate or reuse the immutable report from saved predictions and benchmark labels."""
    # This is label/sequence preparation only; no token cache or model is opened.
    rows = datasets.blf_benchmark_rows(benchmark)
    spec = datasets.DATASETS[benchmark]
    test_rows = np.flatnonzero(rows[spec.split_column].to_numpy() == "test")
    current_context = reporting.benchmark_context(
        rows, task=spec.task, label_column=spec.label_column,
        split_column=spec.split_column,
    )
    inventory = reporting.collect(
        benchmark=benchmark, config=result_recipe(0),
        original_folders=ORIGINAL_RESULTS[benchmark], original_encoders=ORIGINAL,
        swaps=[
            (modality, encoder_key, tokens.grid_resolution(encoder_key) != "codon")
            for modality, encoder_key in SWAPS
        ],
        swap_root=RESULTS_ROOT / benchmark, seeds=COMPARISON_SEEDS,
        selection_original_folders=(
            RESULT_PATHS.selection_originals[benchmark]
            if RESULT_PATHS.fixed_historical_selection else None
        ),
        selection_snapshot=RESULT_PATHS.selection_snapshot,
        selection_swap_root=(
            RESULT_PATHS.selection_swaps / benchmark
            if RESULT_PATHS.fixed_historical_selection else None
        ),
    )
    n_classes = (
        int(rows[spec.label_column].max()) + 1
        if spec.task == "classification" else None
    )
    saved, analysis_path, computed = reporting.report(
        inventory, truth=rows[spec.label_column].to_numpy()[test_rows],
        groups=rows["sequence"].to_numpy()[test_rows], task=spec.task,
        context=current_context, output_dir=COMPARISON_DIR,
        draws=COMPARISON_DRAWS, bootstrap_seed=COMPARISON_BOOTSTRAP_SEED,
        n_classes=n_classes,
        repository=CODE_DIR.parent,
    )
    return saved, analysis_path, computed


In [29]:
SAVED_COMPARISON_REPORTS = {}
if RUN_SAVED_COMPARISONS and not READ_SAVED_RESULTS:
    raise ValueError("Saved-prediction reporting requires READ_SAVED_RESULTS = True.")
from shared_code.fusion import swap_reporting

# Reading verifies existing reports without loading benchmark rows or resampling.
# Ambiguous reports need an explicit file choice; absence never triggers calculation.
for benchmark in BENCHMARKS:
    if EXPERIMENT == "ecoli_ernie":
        print("Focused extension: read the head-score table; single-control reports are not configured.")
        continue
    if RUN_SAVED_COMPARISONS:
        saved, analysis_path, computed = build_or_reuse_report(benchmark, swap_reporting)
        action = "calculated" if computed else "reused"
    elif READ_SAVED_RESULTS:
        expected = dict(config=result_recipe(0), draws=COMPARISON_DRAWS,
                        bootstrap_seed=COMPARISON_BOOTSTRAP_SEED)
        source_roots = None
        if RESULT_PATHS.fixed_historical_selection:
            expected["selection_policy"] = "fixed-historical-seed0-validation"
            # Original report paths stay unchanged; the snapshot maps selection to canonical files.
            source_roots = {
                "outputs": RESULT_PATHS.root / "outputs",
                "historical_results/Code": RESULT_PATHS.root / "historical_results/Code",
            }
        try:
            saved, analysis_path = swap_reporting.read_existing_report(
                benchmark=benchmark, report_dirs=RESULT_PATHS.saved_report_dirs,
                seeds=COMPARISON_SEEDS, expected=expected,
                selected_report=SAVED_REPORT_FILES.get(benchmark), source_roots=source_roots,
                selection_snapshot=RESULT_PATHS.selection_snapshot,
            )
        except (OSError, ValueError) as error:
            print(f"{benchmark}: saved report unavailable: {error}")
            continue
        if saved is None:
            print(f"{benchmark}: no matching saved report; calculation remains disabled.")
            continue
        action = "read"
    else:
        continue
    SAVED_COMPARISON_REPORTS[benchmark] = saved
    analysis = saved["analysis"]
    print(f"{benchmark}: {action} {analysis_path.name} | requested seeds {COMPARISON_SEEDS}")
    summary = pd.DataFrame(analysis["seed_summary"])
    if not summary.empty:
        accuracy_report = analysis["units"] == "accuracy percentage points"
        score_unit = "accuracy (%)" if accuracy_report else "Spearman"
        difference_unit = "percentage points" if accuracy_report else "Spearman"
        columns = {
            "first": "model fit",
            "second": "reference fit",
            "first_score": f"mean model {score_unit}",
            "second_score": f"mean reference {score_unit}",
            "difference": f"mean difference ({difference_unit})",
            "ci_low": f"95% lower bound ({difference_unit})",
            "ci_high": f"95% upper bound ({difference_unit})",
            "seed_sd": f"seed SD ({difference_unit})",
            "status": "coverage",
            "missing_seeds": "missing seeds",
        }
        if "interval_issue" in summary and summary["interval_issue"].fillna("").ne("").any():
            columns["interval_issue"] = "interval issue"
        table = summary.set_index(["encoder", "kind", "comparison"])
        display(table.reindex(columns=list(columns)).rename(columns=columns).round(4))
    if analysis["issues"]:
        display(pd.DataFrame(analysis["issues"]))

cov_vac_degradation: read D:\academic\UCF\2026_fall\Capstone Project CAP6942\Research\genomics-multimodal-fusion\Code\stage4\results\newton_h100_seeds012\local_analyses\paired-report-v1\cov_vac_degradation__9712c98ff132634254ff.json; units: Spearman


first_score  second_score  difference seed_sd  ci_low  ci_high  first_wins  valid_draws  requested_draws interval_issue
encoder  kind           comparison                            seed                                                                                                                         
nt500m   representation replacement single vs original single 0          0.7262        0.7827     -0.0565    None -0.1024  -0.0099      0.0090         2000             2000               
                                                              1          0.7177        0.7946     -0.0769    None -0.1217  -0.0306      0.0010         2000             2000               
                                                              2          0.7249        0.8110     -0.0861    None -0.1292  -0.0450      0.0005         2000             2000               
         replacement    concat vs original concat             0          0.7517        0.8006     -0.0490    None -0.0826  -0.0132      0.0025         2000             2000               
                                                              1          0.7440        0.7980     -0.0540    None -0.0968  -0.0119      0.0080         2000             2000               
...                                                                         ...           ...         ...     ...     ...      ...         ...          ...              ...            ...
protbert method         mil vs concat                         1          0.7924        0.7823      0.0102    None -0.0132   0.0320      0.7995         2000             2000               
                                                              2          0.7719        0.7959     -0.0240    None -0.0503   0.0022      0.0335         2000             2000               
                        cross vs concat                       0          0.8478        0.7886      0.0593    None  0.0291   0.0925      1.0000         2000             2000               
                                                              1          0.8278        0.7823      0.0455    None  0.0193   0.0724      1.0000         2000             2000               
                                                              2          0.8412        0.7959      0.0453    None  0.0162   0.0749      1.0000         2000             2000               

[216 rows x 10 columns]

available_seeds missing_seeds    status  first_score  second_score  difference  seed_sd  ci_low  ci_high  first_wins  valid_draws  \
encoder  kind            comparison                                                                                                                                                                
nt500m   representation  replacement single vs original single       [0, 1, 2]            []  complete       0.7229        0.7961     -0.0732   0.0151 -0.1149  -0.0328      0.0005         2000   
         replacement     concat vs original concat                   [0, 1, 2]            []  complete       0.7463        0.7919     -0.0456   0.0105 -0.0783  -0.0112      0.0040         2000   
         complementarity concat vs validation-selected single        [0, 1, 2]            []  complete       0.7463        0.8373     -0.0910   0.0070 -0.1250  -0.0567      0.0000         2000   
         replacement     mil vs original mil                         [0, 1, 2]            []  complete       0.7049        0.7791     -0.0741   0.0124 -0.1143  -0.0344      0.0000         2000   
         complementarity mil vs validation-selected single           [0, 1, 2]            []  complete       0.7049        0.8373     -0.1324   0.0129 -0.1780  -0.0884      0.0000         2000   
...                                                                        ...           ...       ...          ...           ...         ...      ...     ...      ...         ...          ...   
protbert complementarity mil vs validation-selected single           [0, 1, 2]            []  complete       0.7850        0.8373     -0.0523   0.0161 -0.0865  -0.0179      0.0005         2000   
         replacement     cross vs original cross                     [0, 1, 2]            []  complete       0.8389        0.8349      0.0040   0.0162 -0.0118   0.0206      0.6995         2000   
         complementarity cross vs validation-selected single         [0, 1, 2]            []  complete       0.8389        0.8373      0.0016   0.0075 -0.0243   0.0278      0.5600         2000   
         method          mil vs concat                               [0, 1, 2]            []  complete       0.7850        0.7889     -0.0039   0.0179 -0.0264   0.0173      0.3665         2000   
                         cross vs concat                             [0, 1, 2]            []  complete       0.8389        0.7889      0.0500   0.0080  0.0253   0.0771      1.0000         2000   

                                                                requested_draws interval_issue  
encoder  kind            comparison                                                             
nt500m   representation  replacement single vs original single             2000                 
         replacement     concat vs original concat                         2000                 
         complementarity concat vs validation-selected single              2000                 
         replacement     mil vs original mil                               2000                 
         complementarity mil vs validation-selected single                 2000                 
...                                                                         ...            ...  
protbert complementarity mil vs validation-selected single                 2000                 
         replacement     cross vs original cross                           2000                 
         complementarity cross vs validation-selected single               2000                 
         method          mil vs concat                                     2000                 
                         cross vs concat                                   2000                 

[72 rows x 13 columns]

fungal_expression: read D:\academic\UCF\2026_fall\Capstone Project CAP6942\Research\genomics-multimodal-fusion\Code\stage4\results\newton_h100_seeds012\local_analyses\paired-report-v1\fungal_expression__5dc1468060a1377f0d98.json; units: Spearman


first_score  second_score  difference seed_sd  ci_low  ci_high  first_wins  valid_draws  requested_draws interval_issue
encoder  kind           comparison                            seed                                                                                                                         
nt500m   representation replacement single vs original single 0          0.7397        0.8081     -0.0684    None -0.1059  -0.0354      0.0000         2000             2000               
                                                              1          0.7591        0.8079     -0.0488    None -0.0848  -0.0156      0.0010         2000             2000               
                                                              2          0.7647        0.8148     -0.0501    None -0.0828  -0.0211      0.0000         2000             2000               
         replacement    concat vs original concat             0          0.7866        0.8030     -0.0164    None -0.0417   0.0079      0.0855         2000             2000               
                                                              1          0.7987        0.7902      0.0085    None -0.0189   0.0351      0.7085         2000             2000               
...                                                                         ...           ...         ...     ...     ...      ...         ...          ...              ...            ...
protbert method         mil vs concat                         1          0.7900        0.7952     -0.0052    None -0.0277   0.0161      0.3120         2000             2000               
                                                              2          0.7971        0.7996     -0.0025    None -0.0219   0.0145      0.3760         2000             2000               
                        cross vs concat                       0          0.8215        0.8048      0.0166    None -0.0078   0.0411      0.9165         2000             2000               
                                                              1          0.8199        0.7952      0.0247    None  0.0061   0.0442      0.9970         2000             2000               
                                                              2          0.8062        0.7996      0.0066    None -0.0157   0.0276      0.7120         2000             2000               

[216 rows x 10 columns]

available_seeds missing_seeds    status  first_score  second_score  difference  seed_sd  ci_low  ci_high  first_wins  valid_draws  \
encoder  kind            comparison                                                                                                                                                                
nt500m   representation  replacement single vs original single       [0, 1, 2]            []  complete       0.7545        0.8103     -0.0557   0.0109 -0.0892  -0.0273      0.0000         2000   
         replacement     concat vs original concat                   [0, 1, 2]            []  complete       0.7954        0.7956     -0.0001   0.0141 -0.0241   0.0235      0.4760         2000   
         complementarity concat vs validation-selected single        [0, 1, 2]            []  complete       0.7954        0.7827      0.0127   0.0141 -0.0181   0.0431      0.7950         2000   
         replacement     mil vs original mil                         [0, 1, 2]            []  complete       0.7902        0.7963     -0.0061   0.0054 -0.0296   0.0167      0.2790         2000   
         complementarity mil vs validation-selected single           [0, 1, 2]            []  complete       0.7902        0.7827      0.0075   0.0233 -0.0277   0.0421      0.6585         2000   
...                                                                        ...           ...       ...          ...           ...         ...      ...     ...      ...         ...          ...   
protbert complementarity mil vs validation-selected single           [0, 1, 2]            []  complete       0.7958        0.7441      0.0517   0.0121  0.0219   0.0825      0.9995         2000   
         replacement     cross vs original cross                     [0, 1, 2]            []  complete       0.8158        0.8214     -0.0056   0.0090 -0.0345   0.0225      0.3650         2000   
         complementarity cross vs validation-selected single         [0, 1, 2]            []  complete       0.8158        0.7441      0.0717   0.0112  0.0492   0.0969      1.0000         2000   
         method          mil vs concat                               [0, 1, 2]            []  complete       0.7958        0.7999     -0.0040   0.0014 -0.0218   0.0126      0.3045         2000   
                         cross vs concat                             [0, 1, 2]            []  complete       0.8158        0.7999      0.0160   0.0091 -0.0045   0.0360      0.9445         2000   

                                                                requested_draws interval_issue  
encoder  kind            comparison                                                             
nt500m   representation  replacement single vs original single             2000                 
         replacement     concat vs original concat                         2000                 
         complementarity concat vs validation-selected single              2000                 
         replacement     mil vs original mil                               2000                 
         complementarity mil vs validation-selected single                 2000                 
...                                                                         ...            ...  
protbert complementarity mil vs validation-selected single                 2000                 
         replacement     cross vs original cross                           2000                 
         complementarity cross vs validation-selected single               2000                 
         method          mil vs concat                                     2000                 
                         cross vs concat                                   2000                 

[72 rows x 13 columns]

... (2 outputs left out)

available_seeds missing_seeds    status  first_score  second_score  difference  seed_sd  ci_low  ci_high  first_wins  valid_draws  \
encoder  kind            comparison                                                                                                                                                                
nt500m   representation  replacement single vs original single       [0, 1, 2]            []  complete      48.4082       46.4888      1.9195   1.6756 -1.2640   5.1966      0.8740         2000   
         replacement     concat vs original concat                   [0, 1, 2]            []  complete      57.9588       58.4270     -0.4682   1.6638 -2.5281   1.4993      0.3015         2000   
         complementarity concat vs validation-selected single        [0, 1, 2]            []  complete      57.9588       52.2004      5.7584   1.0969  2.9494   8.5674      1.0000         2000   
         replacement     mil vs original mil                         [0, 1, 2]            []  complete      55.1966       54.5880      0.6086   0.7735 -1.4993   2.9506      0.6810         2000   
         complementarity mil vs validation-selected single           [0, 1, 2]            []  complete      55.1966       52.2004      2.9963   1.9978  0.5150   5.6648      0.9925         2000   
...                                                                        ...           ...       ...          ...           ...         ...      ...     ...      ...         ...          ...   
protbert complementarity mil vs validation-selected single           [0, 1, 2]            []  complete      56.8820       53.4176      3.4644   1.8276  0.5618   6.2734      0.9905         2000   
         replacement     cross vs original cross                     [0, 1, 2]            []  complete      53.3240       55.5243     -2.2004   1.6932 -4.9625   0.5630      0.0640         2000   
         complementarity cross vs validation-selected single         [0, 1, 2]            []  complete      53.3240       53.4176     -0.0936   1.5534 -2.6217   2.4824      0.4690         2000   
         method          mil vs concat                               [0, 1, 2]            []  complete      56.8820       55.0562      1.8258   2.9427 -0.4682   4.2135      0.9355         2000   
                         cross vs concat                             [0, 1, 2]            []  complete      53.3240       55.0562     -1.7322   0.5847 -4.0262   0.7491      0.0835         2000   

                                                                requested_draws interval_issue  
encoder  kind            comparison                                                             
nt500m   representation  replacement single vs original single             2000                 
         replacement     concat vs original concat                         2000                 
         complementarity concat vs validation-selected single              2000                 
         replacement     mil vs original mil                               2000                 
         complementarity mil vs validation-selected single                 2000                 
...                                                                         ...            ...  
protbert complementarity mil vs validation-selected single                 2000                 
         replacement     cross vs original cross                           2000                 
         complementarity cross vs validation-selected single               2000                 
         method          mil vs concat                                     2000                 
                         cross vs concat                                   2000                 

[72 rows x 13 columns]

mrna_stability: read D:\academic\UCF\2026_fall\Capstone Project CAP6942\Research\genomics-multimodal-fusion\Code\stage4\results\newton_h100_seeds012\local_analyses\paired-report-v1\mrna_stability__c807a6c54eea0633a6e2.json; units: Spearman


first_score  second_score  difference seed_sd  ci_low  ci_high  first_wins  valid_draws  requested_draws interval_issue
encoder  kind           comparison                            seed                                                                                                                         
nt500m   representation replacement single vs original single 0          0.5336        0.5363     -0.0026    None -0.0157   0.0099      0.3495         2000             2000               
                                                              1          0.5411        0.5369      0.0042    None -0.0072   0.0154      0.7535         2000             2000               
                                                              2          0.5336        0.5427     -0.0091    None -0.0217   0.0032      0.0745         2000             2000               
         replacement    concat vs original concat             0          0.5493        0.5601     -0.0109    None -0.0211  -0.0006      0.0205         2000             2000               
                                                              1          0.5590        0.5584      0.0007    None -0.0086   0.0095      0.5420         2000             2000               
...                                                                         ...           ...         ...     ...     ...      ...         ...          ...              ...            ...
protbert method         mil vs concat                         1          0.5487        0.5595     -0.0108    None -0.0207  -0.0012      0.0145         2000             2000               
                                                              2          0.5576        0.5597     -0.0021    None -0.0123   0.0081      0.3360         2000             2000               
                        cross vs concat                       0          0.5627        0.5567      0.0060    None -0.0061   0.0171      0.8370         2000             2000               
                                                              1          0.5575        0.5595     -0.0020    None -0.0129   0.0089      0.3770         2000             2000               
                                                              2          0.5487        0.5597     -0.0110    None -0.0213  -0.0009      0.0185         2000             2000               

[216 rows x 10 columns]

available_seeds missing_seeds    status  first_score  second_score  difference  seed_sd  ci_low  ci_high  first_wins  valid_draws  \
encoder  kind            comparison                                                                                                                                                                
nt500m   representation  replacement single vs original single       [0, 1, 2]            []  complete       0.5361        0.5386     -0.0025   0.0067 -0.0123   0.0067      0.3080         2000   
         replacement     concat vs original concat                   [0, 1, 2]            []  complete       0.5541        0.5598     -0.0057   0.0059 -0.0127   0.0013      0.0495         2000   
         complementarity concat vs validation-selected single        [0, 1, 2]            []  complete       0.5541        0.5522      0.0019   0.0101 -0.0080   0.0122      0.6370         2000   
         replacement     mil vs original mil                         [0, 1, 2]            []  complete       0.5540        0.5543     -0.0002   0.0047 -0.0083   0.0081      0.4825         2000   
         complementarity mil vs validation-selected single           [0, 1, 2]            []  complete       0.5540        0.5522      0.0018   0.0064 -0.0094   0.0132      0.6265         2000   
...                                                                        ...           ...       ...          ...           ...         ...      ...     ...      ...         ...          ...   
protbert complementarity mil vs validation-selected single           [0, 1, 2]            []  complete       0.5515        0.5420      0.0095   0.0055 -0.0019   0.0211      0.9495         2000   
         replacement     cross vs original cross                     [0, 1, 2]            []  complete       0.5563        0.5486      0.0077   0.0098 -0.0010   0.0163      0.9530         2000   
         complementarity cross vs validation-selected single         [0, 1, 2]            []  complete       0.5563        0.5420      0.0143   0.0066  0.0031   0.0255      0.9935         2000   
         method          mil vs concat                               [0, 1, 2]            []  complete       0.5515        0.5586     -0.0071   0.0045 -0.0145   0.0002      0.0275         2000   
                         cross vs concat                             [0, 1, 2]            []  complete       0.5563        0.5586     -0.0023   0.0085 -0.0106   0.0062      0.2755         2000   

                                                                requested_draws interval_issue  
encoder  kind            comparison                                                             
nt500m   representation  replacement single vs original single             2000                 
         replacement     concat vs original concat                         2000                 
         complementarity concat vs validation-selected single              2000                 
         replacement     mil vs original mil                               2000                 
         complementarity mil vs validation-selected single                 2000                 
...                                                                         ...            ...  
protbert complementarity mil vs validation-selected single                 2000                 
         replacement     cross vs original cross                           2000                 
         complementarity cross vs validation-selected single               2000                 
         method          mil vs concat                                     2000                 
                         cross vs concat                                   2000                 

[72 rows x 13 columns]

### Newton three-seed findings

The linked Newton reports use 504 fits across four benchmarks and seeds 0, 1, and 2. Each report contains 72 paired contrasts. Earlier seed-0 validation fixes the single-encoder references and the original MIL penalty (0.01). The scores below are examples from those completed runs. The [interpretation record](../../docs/run-records.md#2026-11-16-newton-three-seed-report-interpretation-without-recomputation) identifies the fits and reports behind them.

| Benchmark and replacement | Metric | Validation-selected single and mean score | Concatenation | MIL gated fusion | Cross-attention |
| --- | --- | --- | ---: | ---: | ---: |
| CoV-Vac, RiNALMo | Spearman | RiNALMo native (0.8186) | 0.8145 | 0.8025 | 0.8459 |
| E. coli, RiNALMo | Accuracy | RiNALMo codon grid (56.32%) | 60.11% | 59.08% | 56.37% |
| Fungal expression, mRNA-FM | Spearman | mRNA-FM native (0.8510) | 0.8514 | 0.8324 | 0.8496 |
| mRNA stability, mRNA-FM | Spearman | mRNA-FM native (0.5663) | 0.5677 | 0.5652 | 0.5598 |


#### CoV-Vac: cross-attention beats concatenation

Cross-attention beats concatenation for all eight swaps, with positive differences in all three seeds and paired intervals above zero. RiNALMo cross-attention also improves on its selected native single by +0.0273 [0.0094, 0.0457] Spearman. Whether fusion beats the strongest single in the sweep remains unresolved. CoV-Vac measures chemical degradation of designed RNA, and a protein-mediated explanation would require evidence beyond the modeled translation used here. The [saved CoV-Vac report](results/newton_h100_seeds012/local_analyses/paired-report-v1/cov_vac_degradation__9712c98ff132634254ff.json) records the head and single-encoder contrasts.


#### E. coli: concatenation supplies the clearest fusion gains

Concatenation has a positive interval over the selected single for five of eight swaps. For RiNALMo, it gains +3.79 [1.64, 5.95] percentage points over codon-grid RiNALMo. Cross-attention has a lower mean than concatenation in all eight swaps, with seven intervals entirely below zero. The [saved E. coli report](results/newton_h100_seeds012/local_analyses/paired-report-v1/ecoli_proteins__cf51865eb2ed56b6b4e8.json) gives the contrast against the selected single for each head.


#### Fungal expression: mRNA-FM alone nearly matches fusion

Replacing RNA-FM with mRNA-FM improves concatenation by +0.0558 [0.0310, 0.0809] Spearman. The resulting model gains only +0.0004 [−0.0156, 0.0162] over mRNA-FM alone, leaving the benefit of adding the other streams unresolved. Both contrasts are in the [saved fungal report](results/newton_h100_seeds012/local_analyses/paired-report-v1/fungal_expression__5dc1468060a1377f0d98.json).


#### mRNA stability: little additional benefit beyond mRNA-FM

mRNA-FM concatenation gains +0.0014 [−0.0031, 0.0062] Spearman over mRNA-FM alone. The [saved stability report](results/newton_h100_seeds012/local_analyses/paired-report-v1/mrna_stability__c807a6c54eea0633a6e2.json) uses the supplied split, which shares exact sequences across training and testing. See [grouped stability](grouped_stability.ipynb) for the separate evaluation that keeps exact-sequence groups within partitions.


### What the Newton comparisons establish

The benefit depends on the benchmark, encoder combination, and head. Across the four benchmarks, zero of 32 MIL-versus-concatenation intervals were entirely positive with the inherited penalty of 0.01. Each swap can change pretraining, tokenization, and feature width together. These results measure the whole replacement under the retained BioLangFusion settings.


### Replacing RiNALMo with RNAErnie and ERNIE-RNA

The follow-up compares gating and concatenation on the same 712 E. coli test examples, with six RiNALMo reference fits and twelve new fits. The DNA and protein encoders, split, and head settings carry over. RNAErnie uses the longer-context RNAErnie2 checkpoint. Both new encoders supply 768-wide token states, compared with RiNALMo's 640, so the projections change size. The [checkpoint notes](../../docs/sources.md#rnaernie-and-ernie-rna-checkpoint-implementations) identify the revisions and tokenizer limits.


In [30]:
try:
    if EXPERIMENT == "ecoli_ernie":
        ernie_fits = RESULTS_ROOT
    elif (READ_SAVED_RESULTS and RESULT_PATHS.name == "newton_h100_seeds012"
          and "ecoli_proteins" in BENCHMARKS):
        ernie_fits = resolve_ernie_extension(
            CODE_DIR, workspace=ECOLI_FOLLOWUP_WORKSPACE,
            reference_workspace=FOUR_BENCHMARK_WORKSPACE,
        ).swaps
    else:
        ernie_fits = None

    if ernie_fits is not None:
        seeds = (0, 1, 2) if READ_SAVED_RESULTS else (SEED,)
        display(extension_score_table(ernie_fits, seeds).round(2))
except (OSError, RuntimeError, ValueError, KeyError, TypeError) as missing:
    print("E. coli RNA swap scores unavailable:", missing)

Seed 0 (%)  Seed 1 (%)  Seed 2 (%)  Mean (%)
RNA encoder            head          evidence                                                                
RiNALMo 150M           MIL gated     existing RiNALMo reference       59.41       59.27       58.57     59.08
                       Concatenation existing RiNALMo reference       59.83       61.38       59.13     60.11
RNAErnie (author 2048) MIL gated     focused extension                56.18       59.27       56.32     57.26
                       Concatenation focused extension                57.72       57.72       56.18     57.21
ERNIE-RNA              MIL gated     focused extension                54.92       54.92       56.46     55.43
                       Concatenation focused extension                58.01       56.18       58.43     57.54

RiNALMo has the highest mean accuracy under both heads. For ERNIE-RNA, concatenation exceeds gating in all three seeds, by 2.11 percentage points on average. These are descriptive comparisons of head accuracy. A fusion gain over either new encoder alone would require its single-encoder control. The [saved follow-up results](results/newton_ecoli_ernie/local_analyses/ecoli_ernie_three_seed.json) give the full-precision scores and matched inputs.


### Inspect one training seed

These tables use `SEED` to show each swap's scores, fusion gains (`Δfusion`), and gains over the original combination.


In [27]:
if EXPERIMENT in {"encoder_swaps", "full"}:
    for benchmark in BENCHMARKS:
        units = "accuracy (%) and differences in percentage points" if benchmark in SCALE else "Spearman"
        print(f"\n{benchmark} | {RESULT_PATHS.name} | training seed {SEED} | {units}")
        try:
            display(benchmark_score_table(benchmark).round(3))
        except (OSError, RuntimeError, ValueError, KeyError, TypeError) as missing:
            print("  not ready:", missing)


cov_vac_degradation (Spearman)


alone   best single  best single score  concat  concat Δfusion  concat vs original    mil  mil Δfusion  mil vs original  cross  cross Δfusion  cross vs original
slot encoder                                                                                                                                                                              
none BioLangFusion's own    NaN  unimodal:rna              0.838   0.801          -0.037                 NaN  0.784       -0.053              NaN  0.829         -0.009                NaN
dna  NT 500M human-ref    0.726  unimodal:rna              0.838   0.752          -0.086              -0.049  0.697       -0.141           -0.087  0.813         -0.024             -0.015
     HyenaDNA large       0.755  unimodal:rna              0.838   0.829          -0.009               0.029  0.798       -0.040            0.013  0.862          0.024              0.033
rna  RiNALMo 150M         0.824  unimodal:rna              0.824   0.806          -0.018               0.006  0.818       -0.006            0.034  0.845          0.021              0.016
     mRNA-FM              0.802  unimodal:rna              0.802   0.758          -0.044              -0.042  0.763       -0.040           -0.022  0.830          0.027              0.001
     CaLM                 0.845  unimodal:rna              0.845   0.817          -0.029               0.016  0.787       -0.058            0.002  0.834         -0.011              0.005
prot ESM-2 35M            0.812  unimodal:rna              0.838   0.809          -0.029               0.008  0.793       -0.045            0.008  0.833         -0.005              0.004
     ESM-2 150M           0.827  unimodal:rna              0.838   0.801          -0.037               0.000  0.782       -0.055           -0.002  0.843          0.005              0.014
     ProtBERT             0.776  unimodal:rna              0.838   0.789          -0.049              -0.012  0.791       -0.047            0.006  0.848          0.010              0.019


fungal_expression (Spearman)


alone        best single  best single score  concat  concat Δfusion  concat vs original    mil  mil Δfusion  mil vs original  cross  cross Δfusion  cross vs original
slot encoder                                                                                                                                                                                   
none BioLangFusion's own    NaN       unimodal:dna              0.808   0.803          -0.005                 NaN  0.800       -0.008              NaN  0.819          0.011                NaN
dna  NT 500M human-ref    0.740  unimodal-grid:rna              0.758   0.787           0.028              -0.016  0.790        0.032           -0.010  0.805          0.046             -0.015
     HyenaDNA large       0.736  unimodal-grid:rna              0.758   0.787           0.028              -0.016  0.779        0.021           -0.021  0.803          0.045             -0.016
rna  RiNALMo 150M         0.850       unimodal:rna              0.850   0.846          -0.004               0.043  0.831       -0.019            0.031  0.832         -0.018              0.013
     mRNA-FM              0.852       unimodal:rna              0.852   0.853           0.001               0.050  0.839       -0.013            0.039  0.842         -0.010              0.023
     CaLM                 0.851       unimodal:rna              0.851   0.849          -0.002               0.046  0.848       -0.003            0.048  0.853          0.002              0.034
prot ESM-2 35M            0.695       unimodal:dna              0.808   0.795          -0.013              -0.008  0.780       -0.028           -0.020  0.817          0.009             -0.002
     ESM-2 150M           0.728       unimodal:dna              0.808   0.784          -0.024              -0.019  0.784       -0.024           -0.016  0.818          0.010             -0.001
     ProtBERT             0.761      unimodal:prot              0.761   0.805           0.044               0.002  0.800        0.039            0.000  0.821          0.060              0.002


ecoli_proteins (accuracy x 100)


alone        best single  best single score  concat  concat Δfusion  concat vs original     mil  mil Δfusion  mil vs original   cross  cross Δfusion  cross vs original
slot encoder                                                                                                                                                                                      
none BioLangFusion's own     NaN      unimodal:prot             50.983  59.691           8.708                 NaN  53.933        2.949              NaN  55.618          4.635                NaN
dna  NT 500M human-ref    47.191      unimodal:prot             50.983  57.303           6.320              -2.388  54.916        3.933            0.983  54.354          3.371             -1.264
     HyenaDNA large       50.983      unimodal:prot             50.983  57.725           6.742              -1.966  52.528        1.545           -1.404  54.073          3.090             -1.545
rna  RiNALMo 150M         56.601  unimodal-grid:rna             55.899  59.831           3.933               0.140  59.410        3.511            5.478  55.899          0.000              0.281
     mRNA-FM              59.129       unimodal:rna             59.129  60.112           0.983               0.421  53.652       -5.478           -0.281  55.197         -3.933             -0.421
     CaLM                 57.444       unimodal:rna             57.444  57.022          -0.421              -2.669  56.601       -0.843            2.669  53.652         -3.792             -1.966
prot ESM-2 35M            52.107      unimodal:prot             52.107  56.742           4.635              -2.949  52.528        0.421           -1.404  53.230          1.124             -2.388
     ESM-2 150M           55.758      unimodal:prot             55.758  58.708           2.949              -0.983  56.461        0.702            2.528  53.792         -1.966             -1.826
     ProtBERT             53.511      unimodal:prot             53.511  54.494           0.983              -5.197  58.848        5.337            4.916  53.230         -0.281             -2.388


mrna_stability (Spearman)


alone        best single  best single score  concat  concat Δfusion  concat vs original    mil  mil Δfusion  mil vs original  cross  cross Δfusion  cross vs original
slot encoder                                                                                                                                                                                   
none BioLangFusion's own    NaN       unimodal:rna              0.558   0.560           0.002                 NaN  0.556       -0.002              NaN  0.544         -0.013                NaN
dna  NT 500M human-ref    0.534       unimodal:rna              0.558   0.549          -0.009              -0.011  0.554       -0.004           -0.002  0.546         -0.012              0.002
     HyenaDNA large       0.523       unimodal:rna              0.558   0.556          -0.002              -0.005  0.543       -0.014           -0.012  0.557         -0.001              0.013
rna  RiNALMo 150M         0.561  unimodal-grid:rna              0.554   0.554          -0.001              -0.007  0.560        0.006            0.004  0.549         -0.006              0.004
     mRNA-FM              0.566       unimodal:rna              0.566   0.569           0.003               0.009  0.564       -0.001            0.009  0.563         -0.003              0.018
     CaLM                 0.557       unimodal:rna              0.557   0.557          -0.000              -0.003  0.555       -0.003           -0.001  0.538         -0.020             -0.007
prot ESM-2 35M            0.534       unimodal:rna              0.558   0.553          -0.004              -0.007  0.550       -0.008           -0.006  0.559          0.001              0.015
     ESM-2 150M           0.534       unimodal:rna              0.558   0.561           0.003               0.001  0.548       -0.010           -0.008  0.561          0.003              0.017
     ProtBERT             0.543      unimodal:prot              0.543   0.557           0.014              -0.003  0.548        0.005           -0.008  0.563          0.020              0.018

---

## Supplementary comparisons and run notes


### Historical seed-0 findings

The [October 27 Phase B readout](../../docs/run-records.md#2026-10-27-phaseb-complete-readout-all-eight-swaps-written-up-from-the-saved-notebook-outputs-no-new-fits) records the earlier seed-0 comparisons. Its stability intervals cover three replacements. The fungal example below comes from this historical run.


### RNA-FM and mRNA-FM as a worked encoder comparison

RNA-FM uses nucleotide tokens and was pretrained on noncoding RNA. mRNA-FM uses codon tokens and was pretrained on coding sequences. The [RNA-FM and mRNA-FM sources](../../docs/sources.md#rna-fm-model-card) explain what the authors document about those populations and the limits of the training details available for mRNA-FM.


#### Fungal expression at historical seed 0

RNA-FM scored 0.722 Spearman on native tokens and 0.794 on the codon grid. mRNA-FM scored 0.852 on native tokens. The grid control narrows the gap between the two encoders. The swap changes pretraining, tokenization, width, and capacity together, so these scores describe the combined change.

Replacing RNA-FM with mRNA-FM raised concatenation from 0.793 to 0.853, with a gain over mRNA-FM alone of +0.001 [−0.014, 0.016]. The [saved seed-0 fungal report](results/phaseB/analyses/paired-report-v1/fungal_expression__d2173df49ec55ed37414.json) gives both contrasts. The added fusion gain remains unresolved under that comparison.


### Look up Stage 1 alignment

The table reads raw and composition-controlled linear CKA from the [Stage 1 capture](../../docs/figures/stability/results.json) for the 981-sequence stability sample. Each replacement is compared with the two encoders it joins and with the encoder it displaces. These values describe that sample. Relating them to gains on the four benchmarks would require a separate comparison.

The controlled value for RNA-FM and RiNALMo is 0.486 in the capture and 0.378 in the saved Stage 1 notebook. Treat it as provisional. The [Stage 1 README's discrepancy note](../stage1/README.md#what-carries-into-the-next-stages) points to both records.


In [31]:
def alignment_matrix(capture, name):
    """Restore encoder-labeled axes from the saved JSON matrix."""
    matrix_data = capture["similarity"][name]
    return pd.DataFrame(
        matrix_data["data"], index=matrix_data["index"], columns=matrix_data["columns"],
    )


def alignment_table(capture, configured_swaps):
    """Look up each replacement's Stage 1 similarity to retained and original encoders."""
    cka = alignment_matrix(capture, "Linear CKA")
    cka_controlled = alignment_matrix(capture, "Linear CKA after control")
    alignment = []
    for modality, encoder_key in sorted(configured_swaps):
        triple_keys = {**ORIGINAL, modality: encoder_key}
        # Read similarity to each of the two encoders retained in the triple.
        for other_modality in (slot for slot in ORIGINAL if slot != modality):
            retained_encoder = triple_keys[other_modality]
            alignment.append({
                "swap": ENCODERS[encoder_key].label,
                "pair": f"{ENCODERS[encoder_key].label} + {ENCODERS[retained_encoder].label}",
                "CKA": cka.loc[encoder_key, retained_encoder],
                "CKA after composition control": cka_controlled.loc[encoder_key, retained_encoder],
            })
        # Also compare the replacement with the original encoder it displaced.
        alignment.append({
            "swap": ENCODERS[encoder_key].label,
            "pair": (
                f"{ENCODERS[encoder_key].label} vs the "
                f"{ENCODERS[ORIGINAL[modality]].label} it replaces"
            ),
            "CKA": cka.loc[encoder_key, ORIGINAL[modality]],
            "CKA after composition control": cka_controlled.loc[encoder_key, ORIGINAL[modality]],
        })
    return pd.DataFrame(alignment)


In [32]:
if EXPERIMENT == "ecoli_ernie":
    print("The two new RNA encoders have no Stage 1 alignment capture; this lookup is skipped.")
else:
    alignment = []
    try:
        # Reuse the Stage 1 capture; no alignment is recomputed for these benchmarks.
        stage1 = json.loads(STAGE1_ALIGNMENT.read_text())
        alignment = alignment_table(stage1, set(SWAPS))
        display(alignment.set_index(["swap", "pair"]).round(3))
    except (OSError, ValueError, KeyError, TypeError) as exc:
        if not READ_SAVED_RESULTS:
            raise
        print(f"Stage 1 alignment unavailable: {exc}. No alignment was recalculated.")


CKA  CKA after composition control
swap              pair                                                                                    
HyenaDNA large    HyenaDNA large + RNA-FM                             0.398                          0.228
                  HyenaDNA large + ESM-2 8M                           0.161                          0.109
                  HyenaDNA large vs the NT v2 100M multi-species ...  0.736                          0.117
NT 500M human-ref NT 500M human-ref + RNA-FM                          0.530                          0.196
                  NT 500M human-ref + ESM-2 8M                        0.163                          0.126
                  NT 500M human-ref vs the NT v2 100M multi-speci...  0.398                          0.064
ESM-2 150M        ESM-2 150M + NT v2 100M multi-species               0.067                          0.030
                  ESM-2 150M + RNA-FM                                 0.200                          0.104
                  ESM-2 150M vs the ESM-2 8M it replaces              0.553                          0.391
ESM-2 35M         ESM-2 35M + NT v2 100M multi-species                0.083                          0.045
                  ESM-2 35M + RNA-FM                                  0.199                          0.175
                  ESM-2 35M vs the ESM-2 8M it replaces               0.744                          0.729
ProtBERT          ProtBERT + NT v2 100M multi-species                 0.058                          0.027
                  ProtBERT + RNA-FM                                   0.116                          0.083
                  ProtBERT vs the ESM-2 8M it replaces                0.199                          0.085
CaLM              CaLM + NT v2 100M multi-species                     0.183                          0.058
                  CaLM + ESM-2 8M                                     0.433                          0.421
                  CaLM vs the RNA-FM it replaces                      0.632                          0.194
mRNA-FM           mRNA-FM + NT v2 100M multi-species                  0.030                          0.044
                  mRNA-FM + ESM-2 8M                                  0.046                          0.106
                  mRNA-FM vs the RNA-FM it replaces                   0.126                          0.206
RiNALMo 150M      RiNALMo 150M + NT v2 100M multi-species             0.209                          0.089
                  RiNALMo 150M + ESM-2 8M                             0.339                          0.227
                  RiNALMo 150M vs the RNA-FM it replaces              0.809                          0.486

### Optional single-seed paired intervals

This earlier workflow compares each fusion head with the original head and the selected single at `SEED`. It checks saved predictions against the current test-target order and resamples exact test-sequence groups using 2,000 draws and resampling seed 0. Matching targets alone cannot establish row identity for older fits with incomplete provenance. The `resolved` column marks intervals that exclude zero in either direction.

To calculate these intervals, set `RUN_SINGLE_SEED_INTERVALS = True` and `READ_SAVED_RESULTS = False`. Earlier cells can then prepare caches and, if `RUN_TRAINING = True`, fit pending models. Use the main report above for paired comparisons across seeds and head comparisons against concatenation.


In [33]:
def bootstrap_comparison_plan(benchmark):
    """Name the saved records and eligible original/best-single comparisons."""
    original = results_for_encoder_set(benchmark)
    records, comparisons = {}, []

    for name, result in original.items():
        records[f"original {name}"] = result
    for modality, encoder_key in SWAPS:
        runs = results_for_encoder_set(benchmark, modality, encoder_key)
        selection_runs = (
            runs if SEED == SELECTION_SEED
            else results_for_encoder_set(
                benchmark, modality, encoder_key, seed=SELECTION_SEED,
            )
        )
        selected_single = validation_selected_single(
            runs, modality, encoder_key, selection_runs, benchmark=benchmark,
        )
        for name, result in runs.items():
            records[f"{encoder_key} {name}"] = result
        for head in HEADS:
            if head not in runs or head not in original:
                continue
            # Each tuple names the displayed comparison, first predictor, and reference.
            comparisons.append((
                f"{encoder_key} {head} vs original",
                f"{encoder_key} {head}",
                f"original {head}",
            ))
            if selected_single is not None:
                comparisons.append((
                    f"{encoder_key} {head} vs best single ({selected_single})",
                    f"{encoder_key} {head}",
                    f"{encoder_key} {selected_single}",
                ))
    return records, comparisons


In [34]:
def paired_bootstrap_differences(benchmark, draws=2000):
    """Compare saved predictions on paired, sequence-grouped test resamples."""
    # Prediction pairing needs the benchmark's labels/order, not any token arrays.
    rows = BENCHMARK_ROWS.get(benchmark)
    if rows is None:
        rows = datasets.blf_benchmark_rows(benchmark)
    spec = datasets.DATASETS[benchmark]
    test = np.flatnonzero(rows[spec.split_column].to_numpy() == "test")
    truth = rows[spec.label_column].to_numpy()[test]
    task = spec.task
    target_dtype = np.int64 if task == "classification" else np.float32
    saved_target_order = np.asarray(truth, dtype=target_dtype).tolist()
    groups = rows["sequence"].to_numpy()[test]
    n_outputs = int(rows[spec.label_column].max()) + 1 if task == "classification" else None
    records, comparisons = bootstrap_comparison_plan(benchmark)
    predictions = {}

    # Validate only records used by a comparison: score-only historical baselines
    # remain readable, but missing/misordered predictions cannot support a paired interval.
    needed = {name for _, first, second in comparisons for name in (first, second)}
    for name in sorted(needed):
        result = records[name]
        issue = swap_results.prediction_issue(
            vars(result), saved_target_order, task=task,
            n_classes=n_outputs,
        )
        if issue:
            raise ValueError(f"{benchmark}, {name}: {issue}; paired comparison not calculated")
        predictions[name] = [np.asarray(result.test_prediction)]
    if not comparisons:
        raise ValueError(f"{benchmark}: no compatible paired comparisons; inspect result coverage")

    # Each prediction list contains one fitted run; this is not a multi-seed average here.
    out = pd.DataFrame(
        train.seed_averaged_bootstrap(
            task,
            truth,
            predictions,
            comparisons,
            groups=groups,
            draws=draws,
            seed=0,
        )
    )

    # Convert score differences to percentage points for classification, not first_wins.
    scale = SCALE.get(benchmark, 1.0)
    for column in ("difference", "ci_low", "ci_high"):
        out[column] = out[column] * scale
    # Excluding zero resolves either a gain or a loss; retain the difference's sign.
    out["resolved"] = np.where(
        (out["ci_low"] > 0) | (out["ci_high"] < 0),
        "yes",
        "no",
    )
    return out.set_index("comparison")[[
        "difference", "ci_low", "ci_high", "first_wins", "resolved",
    ]]

In [35]:
RESOLVED = {}
if EXPERIMENT == "ecoli_ernie" or READ_SAVED_RESULTS or not RUN_SINGLE_SEED_INTERVALS:
    print("Single-seed intervals were skipped. Retained interval outputs are historical.")
else:
    for benchmark in BENCHMARKS:
        try:
            RESOLVED[benchmark] = paired_bootstrap_differences(benchmark)
        except (RuntimeError, ValueError, KeyError) as missing:
            print(f"{benchmark}: not ready ({missing})")
            continue
        print(f"\n{benchmark}")
        display(RESOLVED[benchmark].round(3))


Single-seed intervals were skipped. Retained interval outputs are historical.
